In [13]:
import os
import json
from pathlib import Path
import re
import shutil
import pandas as pd
from utils import main_path

The following function prepares the folder for the results data that will be evaluated 

In [2]:
def create_final_world_states_folder(new_results_path):
    """Copy the final world state of each structure, rename them with the structure id and move them to a new folder in final_world_states, based on the input format"""
    structures_id = ""
    file_type = ".json"
    for dir_name in os.listdir(new_results_path):   # Iterate over each directory in new_results_path and retain its name (structure id)
        if os.path.isdir(os.path.join(new_results_path, dir_name)):
                structures_id = dir_name + file_type
        for subf in os.listdir(os.path.join(new_results_path, dir_name)):   # Iterate over the input format subfolders in the directory, copy the final world state file, 
                                                                            # rename it with the structure id and move it to the new dedicated folder
            if subf == "img_only":
                json_files = [
                    f for f in os.listdir(os.path.join(new_results_path, dir_name, subf))
                    if os.path.isfile(os.path.join(new_results_path, dir_name, subf, f)) and f == "final_world_state.json"
                ]

                #json_files.sort(key = lambda x: int(re.findall(r'\d+', x)[0]) if re.findall (r'\d+', x) else 0)
                #final_world_state = json_files[-1]
                final_world_state_path = os.path.join(new_results_path, dir_name, subf, json_files[0])
                src_path = Path(final_world_state_path)
                dst_path = Path(os.path.join(main_path, "results", "final_world_states", "img_only"))
                new_path = dst_path / structures_id
                shutil.copy2(src_path, new_path)
            elif subf == "json_only":
                json_files = [
                    f for f in os.listdir(os.path.join(new_results_path, dir_name, subf))
                    if os.path.isfile(os.path.join(new_results_path, dir_name, subf, f)) and f == "final_world_state.json"
                ]
                
                #json_files.sort(key = lambda x: int(re.findall(r'\d+', x)[0]) if re.findall (r'\d+', x) else 0)
                #final_world_state = json_files[-1]
                final_world_state_path = os.path.join(new_results_path, dir_name, subf, json_files[0])
                src_path = Path(final_world_state_path)
                dst_path = Path(os.path.join(main_path, "results", "final_world_states", "json_only"))
                new_path = dst_path / structures_id
                shutil.copy2(src_path, new_path)
            elif subf == "img_json":
                json_files = [
                    f for f in os.listdir(os.path.join(new_results_path, dir_name, subf))
                    if os.path.isfile(os.path.join(new_results_path, dir_name, subf, f)) and f == "final_world_state.json"
                ]

                #json_files.sort(key = lambda x: int(re.findall(r'\d+', x)[0]) if re.findall (r'\d+', x) else 0)
                #final_world_state = json_files[-1]
                final_world_state_path = os.path.join(new_results_path, dir_name, subf, json_files[0])
                src_path = Path(final_world_state_path)
                dst_path = Path(os.path.join(main_path, "results", "final_world_states", "img_json"))
                new_path = dst_path / structures_id
                shutil.copy2(src_path, new_path)


new_results_path = os.path.join(main_path, "results", "new_results")
create_final_world_states_folder(new_results_path)

The actual evaluation process starts here

In [ ]:
def parse_structure(json_data):
    """
    Convert JSON data (either a direct list of block dictionaries
    or a dictionary containing a 'dialogue' key with block dictionaries)
    into a dictionary with (x, y, z) tuples as keys and block_color as values.
    """
    block_list_to_parse = []

    if isinstance(json_data, dict) and "dialogue" in json_data:
        block_list_to_parse = json_data["dialogue"]
    elif isinstance(json_data, list):
        block_list_to_parse = json_data
    else:
        print(f"Warning: Unexpected JSON data format encountered in parse_structure. Type: {type(json_data)}. Expected list or dict with 'dialogue' key.")
        return {} 


    return {(block["x"], block["y"], block["z"]): block["block_color"] for block in block_list_to_parse}

In [4]:
def normalize_structure_coords(structure):
    """Shift all block coordinates so the structure starts at (0, 0, 0), to avoid penalties for global shifts."""
    if not structure:
        return {}
    
    # Find the minimum x, y, z coordinates to understand how shifted the structure is w.r.t. the origin
    min_x = min(coord[0] for coord in structure)
    min_y = min(coord[1] for coord in structure)
    min_z = min(coord[2] for coord in structure)

    # Normalize by subtracting the minimum values so that the smallest coordinate becomes (0, 0, 0) and the other coordinates are modified accordingly
    normalized = {
        (x - min_x, y - min_y, z - min_z): color
        for (x, y, z), color in structure.items()
    }
    return normalized

In [ ]:
# Rotation Functions 


def rotate_x_90(x, y, z):
    """Rotate +90 degrees around the X-axis."""
    return (x, -z, y)

def rotate_y_90(x, y, z):
    """Rotate +90 degrees around the Y-axis."""
    return (z, y, -x)

def rotate_z_90(x, y, z):
    """Rotate +90 degrees around the Z-axis."""
    return (-y, x, z)

def apply_rotation_to_structure(structure, rotation_function):
    """
    Applies a given rotation function to all blocks in a structure.
    Returns a new dictionary with rotated coordinates.
    """
    rotated_structure = {}
    for (x, y, z), color in structure.items():
        new_x, new_y, new_z = rotation_function(x, y, z)
        rotated_structure[(new_x, new_y, new_z)] = color
    return rotated_structure

In [ ]:
# Define all 24 90-degree rotations

ROTATION_FUNCTIONS = [
    lambda x, y, z: (x, y, z),       # Identity
    rotate_x_90,                     # X+90
    lambda x, y, z: (x, -y, -z),     # X+180
    lambda x, y, z: (x, z, -y),      # X+270

    rotate_y_90,                     # Y+90
    lambda x, y, z: (-x, y, -z),     # Y+180
    lambda x, y, z: (-z, y, x),      # Y+270

    rotate_z_90,                     # Z+90
    lambda x, y, z: (-x, -y, z),     # Z+180
    lambda x, y, z: (y, -x, z),      # Z+270

    # Combinations for remaining 14 orientations
    lambda x, y, z: rotate_x_90(*rotate_y_90(x,y,z)), # Y+90, X+90
    lambda x, y, z: rotate_x_90(*rotate_y_90(*rotate_y_90(x,y,z))), # Y+180, X+90
    lambda x, y, z: rotate_x_90(*rotate_y_90(*rotate_y_90(*rotate_y_90(x,y,z)))), # Y+270, X+90
    
    lambda x, y, z: rotate_x_90(*rotate_z_90(x,y,z)), # Z+90, X+90
    lambda x, y, z: rotate_x_90(*rotate_z_90(*rotate_z_90(x,y,z))), # Z+180, X+90
    lambda x, y, z: rotate_x_90(*rotate_z_90(*rotate_z_90(*rotate_z_90(x,y,z)))), # Z+270, X+90

    lambda x, y, z: rotate_y_90(*rotate_x_90(x,y,z)), # X+90, Y+90
    lambda x, y, z: rotate_y_90(*rotate_x_90(*rotate_x_90(x,y,z))), # X+180, Y+90
    lambda x, y, z: rotate_y_90(*rotate_x_90(*rotate_x_90(*rotate_x_90(x,y,z)))), # X+270, Y+90

    lambda x, y, z: rotate_y_90(*rotate_z_90(x,y,z)), # Z+90, Y+90
    lambda x, y, z: rotate_y_90(*rotate_z_90(*rotate_z_90(x,y,z))), # Z+180, Y+90
    lambda x, y, z: rotate_y_90(*rotate_z_90(*rotate_z_90(*rotate_z_90(x,y,z)))), # Z+270, Y+90

    lambda x, y, z: rotate_z_90(*rotate_x_90(x,y,z)), # X+90, Z+90
    lambda x, y, z: rotate_z_90(*rotate_y_90(x,y,z)), # Y+90, Z+90
]

In [ ]:
# Helper function for normalisation and comparison 


def normalize_and_compare(target_struct_dict, generated_struct_dict):
    """
    Finds the best rotational alignment of the generated structure to the target structure,
    and returns the Jaccard similarity and block-level TP, FP, FN counts for that best alignment.
    """
    if not target_struct_dict and not generated_struct_dict:
        return 1.0, 0, 0, 0 # Jaccard, TP, FP, FN for blocks
    if not target_struct_dict: # Generated exists, target doesn't
        return 0.0, 0, len(generated_struct_dict), 0
    if not generated_struct_dict: # Target exists, generated doesn't
        return 0.0, 0, 0, len(target_struct_dict)

    best_jaccard = -1
    best_tp_blocks = 0
    best_fp_blocks = 0
    best_fn_blocks = 0
    
    # Normalize target structure once (to its own origin)
    normalized_target_dict = normalize_structure_coords(target_struct_dict)

    # Iterate through all 24 possible rotations for the generated structure
    for rot_fn in ROTATION_FUNCTIONS:
        # Apply rotation to the generated structure
        rotated_generated_dict = apply_rotation_to_structure(generated_struct_dict, rot_fn)
        # Normalize the rotated generated structure (to its own origin)
        normalized_rotated_generated_dict = normalize_structure_coords(rotated_generated_dict)

        # Calculate block-level TP, FP, FN for this rotation 
        tp_blocks_current = 0
        
        # True Positives: Blocks present in both, with same coordinates and same color
        for coords, color in normalized_rotated_generated_dict.items():
            if coords in normalized_target_dict and normalized_target_dict[coords] == color:
                tp_blocks_current += 1
        
        # False Positives: Blocks in generated that are not in target (or wrong color)
        fp_blocks_current = len(normalized_rotated_generated_dict) - tp_blocks_current
        
        # False Negatives: Blocks in target that are not in generated (or wrong color)
        fn_blocks_current = len(normalized_target_dict) - tp_blocks_current

        # Calculate Jaccard for this rotation
        union_size = tp_blocks_current + fp_blocks_current + fn_blocks_current
        jaccard = tp_blocks_current / union_size if union_size > 0 else 0

        if jaccard > best_jaccard:
            best_jaccard = jaccard
            best_tp_blocks = tp_blocks_current
            best_fp_blocks = fp_blocks_current
            best_fn_blocks = fn_blocks_current
            
    # Return best Jaccard along with the corresponding TP, FP, FN block counts
    return best_jaccard, best_tp_blocks, best_fp_blocks, best_fn_blocks

In [ ]:
# Wrapper to get overall similarity 


def compute_structure_metrics(target_struct_dict, generated_struct_dict):
    """
    Computes a similarity score (Jaccard Index) and block-level TP/FP/FN counts
    between two structures, considering all their canonical orientations.
    """
    return normalize_and_compare(target_struct_dict, generated_struct_dict)

In [ ]:
# Main Evaluation Function 


def evaluate_structures_and_metrics(results_dir, targets_dir, experimental_condition_name, main_path):
    """
    Evaluate the built structures against the target structures for a specific experimental condition,
    calculate block-wise metrics, and store detailed results.
    """
    results_map = {
        f.strip().lower(): os.path.join(results_dir, f)
        for f in os.listdir(results_dir)
        if f.endswith(".json") and os.path.isfile(os.path.join(results_dir, f))
    }

    targets_map = {}
    for root, _, files in os.walk(targets_dir): 
        for f in files:
            if f.endswith(".json"):
                clean_name = f.strip().lower()
                full_path = os.path.join(root, f)
                targets_map[clean_name] = full_path

    # Match files by name 
    matching_names = results_map.keys() & targets_map.keys()
    
    if not matching_names:
        print(f" No matching files found for {os.path.basename(os.path.normpath(results_dir))}")
        return {
            "num_generated_structures": len(results_map),
            "num_target_structures": len(targets_map),
            "total_block_tp": 0, "total_block_fp": 0, "total_block_fn": 0,
            "overall_block_precision": 0.0, "overall_block_recall": 0.0, "overall_block_f1_score": 0.0,
            "details": {}
        }

    all_results_with_scores = {}
    
    # Initialize totals for aggregate block-level metrics
    total_block_tp_sum = 0
    total_block_fp_sum = 0
    total_block_fn_sum = 0

    # Process matched pairs 
    for name in sorted(matching_names):
        result_path = results_map[name]
        target_path = targets_map[name]

        with open(result_path, 'r', encoding='utf-8') as rf, \
             open(target_path, 'r', encoding='utf-8') as tf:
            try: 
                result_data_raw = json.load(rf)
                target_data_raw = json.load(tf)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON for {name}: {e}. Skipping this pair.")
                continue 

        # Parse to dict keyed by coordinates
        result_struct_dict = parse_structure(result_data_raw)
        target_struct_dict = parse_structure(target_data_raw)

        # Compute Jaccard and block-level counts (TP, FP, FN) for the optimal rotation
        jaccard_score, tp_blocks, fp_blocks, fn_blocks = \
            compute_structure_metrics(target_struct_dict, result_struct_dict)
        
        # Calculate block-level precision, recall, f1 for this specific structure
        block_precision = tp_blocks / (tp_blocks + fp_blocks) if (tp_blocks + fp_blocks) > 0 else 0
        block_recall = tp_blocks / (tp_blocks + fn_blocks) if (tp_blocks + fn_blocks) > 0 else 0
        block_f1_score = (2 * block_precision * block_recall) / (block_precision + block_recall) \
                         if (block_precision + block_recall) > 0 else 0
        
        # Accumulate total block counts for overall averages
        total_block_tp_sum += tp_blocks
        total_block_fp_sum += fp_blocks
        total_block_fn_sum += fn_blocks

        # Store detailed results for this structure
        all_results_with_scores[name] = {
            "jaccard_similarity": jaccard_score, 
            "generated_structure_path": result_path,
            "target_structure_path": target_path,
            "block_level_tp": tp_blocks,
            "block_level_fp": fp_blocks,
            "block_level_fn": fn_blocks,
            "block_level_precision": block_precision,
            "block_level_recall": block_recall,
            "block_level_f1_score": block_f1_score,
            "generated_data_preview": result_data_raw[:5] if isinstance(result_data_raw, list) else "..."
        }
        print(f"File: {name}, Jaccard: {jaccard_score:.2f}, Block P/R/F1: {block_precision:.2f}/{block_recall:.2f}/{block_f1_score:.2f}")

    # Account for False Negatives (targets that were not generated) 
    unmatched_target_ids = set(targets_map.keys()) - set(matching_names)
    for name in sorted(unmatched_target_ids):
        try:
            with open(targets_map[name], 'r', encoding='utf-8') as tf:
                target_data_raw = json.load(tf)
            num_target_blocks_missing = len(parse_structure(target_data_raw))
        except json.JSONDecodeError:
            num_target_blocks_missing = 0 # Cannot parse target, assume 0 for FN count
        
        total_block_fn_sum += num_target_blocks_missing # Add all blocks from missing targets to FN

        all_results_with_scores[name] = {
            "jaccard_similarity": 0.0, # Cannot compute Jaccard for missing generated structure
            "status": "NOT_GENERATED", 
            "target_structure_path": targets_map[name],
            "generated_structure_path": None,
            "block_level_tp": 0,
            "block_level_fp": 0,
            "block_level_fn": num_target_blocks_missing,
            "block_level_precision": 0.0, # No generated blocks, so precision is 0
            "block_level_recall": 0.0,    # All target blocks are missing, so recall is 0
            "block_level_f1_score": 0.0,
            "generated_data_preview": None
        }
        print(f"File: {name}, Status: NOT_GENERATED")

    # Account for additional False Positives (generated structures without a target) 
    unmatched_generated_ids = set(results_map.keys()) - set(targets_map.keys())
    for name in sorted(unmatched_generated_ids):
        try:
            with open(results_map[name], 'r', encoding='utf-8') as f:
                generated_preview_data = json.load(f)
                num_generated_blocks_unexpected = len(parse_structure(generated_preview_data))
                generated_preview_data = generated_preview_data[:5] if isinstance(generated_preview_data, list) else "..."
        except json.JSONDecodeError:
            num_generated_blocks_unexpected = 0
            generated_preview_data = "Error loading JSON for preview"

        total_block_fp_sum += num_generated_blocks_unexpected # Add all blocks from unexpected generated to FP

        all_results_with_scores[name] = {
            "jaccard_similarity": 0.0, # Cannot compute Jaccard without target
            "status": "UNEXPECTED_GENERATED", 
            "generated_structure_path": results_map[name],
            "target_structure_path": None,
            "block_level_tp": 0,
            "block_level_fp": num_generated_blocks_unexpected,
            "block_level_fn": 0,
            "block_level_precision": 0.0, # No correct blocks (no target), so precision is 0
            "block_level_recall": 0.0,    # No target blocks to miss, so recall is N/A (set to 0 for consistency)
            "block_level_f1_score": 0.0,
            "generated_data_preview": generated_preview_data
        }
        print(f"File: {name}, Status: UNEXPECTED_GENERATED")


    # Calculate overall aggregate block-level metrics for this experimental condition
    overall_block_precision = total_block_tp_sum / (total_block_tp_sum + total_block_fp_sum) if (total_block_tp_sum + total_block_fp_sum) > 0 else 0
    overall_block_recall = total_block_tp_sum / (total_block_tp_sum + total_block_fn_sum) if (total_block_tp_sum + total_block_fn_sum) > 0 else 0
    overall_block_f1_score = 2 * (overall_block_precision * overall_block_recall) / (overall_block_precision + overall_block_recall) if (overall_block_precision + overall_block_recall) > 0 else 0

    metrics = {
        "num_generated_structures": len(results_map),
        "num_target_structures": len(targets_map),
        "total_block_tp": total_block_tp_sum,
        "total_block_fp": total_block_fp_sum,
        "total_block_fn": total_block_fn_sum,
        "overall_block_precision": overall_block_precision,
        "overall_block_recall": overall_block_recall,
        "overall_block_f1_score": overall_block_f1_score
    }
    
    print(f"\nOverall Block-level Metrics for {experimental_condition_name}:")
    print(f" Total TP Blocks: {total_block_tp_sum}, FP Blocks: {total_block_fp_sum}, FN Blocks: {total_block_fn_sum}")
    print(f" Overall Block Precision: {overall_block_precision:.2f}")
    print(f" Overall Block Recall: {overall_block_recall:.2f}")
    print(f" Overall Block F1-Score: {overall_block_f1_score:.2f}")

    # Save detailed results for this condition
    output_filename = f"evaluation_results_{experimental_condition_name}.json"
    analysis_output_path = os.path.join(main_path, "analysis", output_filename)
    os.makedirs(os.path.dirname(analysis_output_path), exist_ok=True) 
    
    with open(analysis_output_path, 'w', encoding='utf-8') as outfile:
        json.dump({"metrics": metrics, "details": all_results_with_scores}, outfile, indent=2)
    print(f"Detailed results for {experimental_condition_name} saved to {analysis_output_path}")

    return metrics # Return the metrics for aggregation if evaluating multiple conditions


In [ ]:
# Main execution loop for experimental conditions 


if __name__ == '__main__':
    targets_base_dir = os.path.join(main_path, "data", "structures", "20_new_project")
    results_base_dir = os.path.join(main_path, "results", "final_world_states") 
    analysis_output_dir = os.path.join(main_path, "analysis")

    # Ensure analysis output directory exists
    os.makedirs(analysis_output_dir, exist_ok=True)

    
    experimental_conditions = ["img_only", "json_only", "img_json"] 

    all_conditions_metrics = {}

    for condition_name in experimental_conditions:
        current_results_dir = os.path.join(results_base_dir, condition_name)
        
        # Check if the results directory for the condition exists
        if not os.path.isdir(current_results_dir):
            print(f"\n--- Skipping Condition: {condition_name} ---")
            print(f"Results directory not found: {current_results_dir}")
            continue # Skip to the next condition

        print(f"\n--- Evaluating Condition: {condition_name} ---")
        
        metrics = evaluate_structures_and_metrics(
            results_dir=current_results_dir,
            targets_dir=targets_base_dir,
            experimental_condition_name=condition_name,
            main_path=main_path # Pass the imported main_path
        )
        all_conditions_metrics[condition_name] = metrics

    print("\n--- Overall Metrics Across Conditions ---")
    # This section aggregates the metrics from each condition
    for condition, metrics in all_conditions_metrics.items():
        print(f"\nCondition: {condition}")
        print(f" Number of Generated Structures: {metrics['num_generated_structures']}")
        print(f" Number of Target Structures: {metrics['num_target_structures']}")
        print(f" Total TP Blocks: {metrics['total_block_tp']}, FP Blocks: {metrics['total_block_fp']}, FN Blocks: {metrics['total_block_fn']}")
        print(f" Overall Block Precision: {metrics['overall_block_precision']:.2f}")
        print(f" Overall Block Recall: {metrics['overall_block_recall']:.2f}")
        print(f" Overall Block F1-Score: {metrics['overall_block_f1_score']:.2f}")

    # Save aggregated metrics
    aggregated_metrics_path = os.path.join(analysis_output_dir, "aggregated_metrics_new.json")
    with open(aggregated_metrics_path, 'w', encoding='utf-8') as outfile:
        json.dump(all_conditions_metrics, outfile, indent=2)
    print(f"\nAggregated metrics saved to {aggregated_metrics_path}")

In [ ]:
# Data analysis


def analyze_and_visualize_scores(main_path):
    """
    Concatenates score data from multiple JSON files, converts to DataFrame,
    computes, and visualizes mean scores per condition.
    """
    analysis_output_dir = os.path.join(main_path, "analysis")
    
    
    experimental_conditions = ["img_only", "json_only", "img_json"] 

    all_detailed_results = []

    print("--- Loading and Concatenating Data ---")
    for condition_name in experimental_conditions:
        filepath = os.path.join(analysis_output_dir, f"evaluation_results_{condition_name}.json")
        
        if not os.path.exists(filepath):
            print(f"Warning: File not found for condition '{condition_name}': {filepath}. Skipping.")
            continue
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                detailed_scores = data.get("details", {})
                
                for structure_id, scores_dict in detailed_scores.items():
                    # Create a copy 
                    score_entry = scores_dict.copy()
                    score_entry['structure_id'] = structure_id # Add structure ID as a column
                    score_entry['condition'] = condition_name    # Add condition name as a column
                    all_detailed_results.append(score_entry)
            print(f"Successfully loaded data for condition: {condition_name}")

        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from {filepath}: {e}. Skipping.")
        except Exception as e:
            print(f"An unexpected error occurred while processing {filepath}: {e}. Skipping.")

    if not all_detailed_results:
        print("No data loaded. Exiting.")
        return

    print("\n--- Converting to Pandas DataFrame ---")
    df = pd.DataFrame(all_detailed_results)
    
    # Drop columns that are not scores or identifiers
    score_columns = [
        "jaccard_similarity",
        "block_level_tp",
        "block_level_fp",
        "block_level_fn",
        "block_level_precision",
        "block_level_recall",
        "block_level_f1_score"
    ]
    
    # Filter DataFrame to include only relevant columns for mean calculation
    numeric_score_df = df[score_columns].apply(pd.to_numeric, errors='coerce')
    
    # Re-add 'condition' for grouping
    numeric_score_df['condition'] = df['condition']

    print(f"DataFrame created with {len(df)} entries.")

    print("\n--- Computing Mean Scores per Condition ---")
    
    # Group by 'condition' and compute the mean for all numeric score columns
    mean_scores_per_condition = numeric_score_df.groupby('condition').mean()

    print("\n--- Mean Scores per Condition (DataFrame Visualization) ---")
    print(mean_scores_per_condition.round(4)) 

if __name__ == '__main__':
    analyze_and_visualize_scores(main_path)

In [8]:
# Data analysis for judge's evaluation

def compute_judge_score_means(json_path):
    # Load the JSON data
    with open(json_path, "r") as infile:
        data = json.load(infile)

    # Filter and collect rows with valid numeric judge evaluations
    rows = []
    for item in data:
        try:
            score = float(item["judge_evaluation"])
            rows.append({
                "structure_id": item["structure_id"],
                "condition": item["condition"],
                "judge_score": score
            })
        except (ValueError, TypeError):
            continue  # skip if not a valid number

    # Create a DataFrame
    df = pd.DataFrame(rows)

    # Compute means by condition
    means_by_condition = df.groupby("condition")["judge_score"].mean().reset_index()

    # Add overall mean as a row
    overall_mean = pd.DataFrame([{
        "condition": "Overall",
        "judge_score": df["judge_score"].mean()
    }])

    # Combine results
    result_df = pd.concat([means_by_condition, overall_mean], ignore_index=True)

    return result_df

# Example usage
result = compute_judge_score_means("/home/xmakaco/cimec/MLLMs-construction-company-main-new/results/all_dialogues_judged/all_dialogues_judged.json")
print(result)


   condition  judge_score
0   img_json     2.650000
1   img_only     2.800000
2  json_only     2.550000
3    Overall     2.666667


In [ ]:
# Data analysis IoU + Block wise

import os
import json
import pandas as pd
from utils import main_path 

def analyze_and_visualize_scores(main_path):
    """
    Concatenates score data from multiple JSON files, converts to DataFrame,
    computes, and visualizes mean scores per condition.
    """
    analysis_output_dir = os.path.join(main_path, "analysis")
    
    
    experimental_conditions = ["img_only", "json_only", "img_json"] 

    all_detailed_results = []

    print("--- Loading and Concatenating Data ---")
    for condition_name in experimental_conditions:
        filepath = os.path.join(analysis_output_dir, f"evaluation_results_{condition_name}.json")
        
        if not os.path.exists(filepath):
            print(f"Warning: File not found for condition '{condition_name}': {filepath}. Skipping.")
            continue
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
                detailed_scores = data.get("details", {})
                
                for structure_id, scores_dict in detailed_scores.items():
                    # Create a copy 
                    score_entry = scores_dict.copy()
                    score_entry['structure_id'] = structure_id # Add structure ID as a column
                    score_entry['condition'] = condition_name    # Add condition name as a column
                    all_detailed_results.append(score_entry)
            print(f"Successfully loaded data for condition: {condition_name}")

        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from {filepath}: {e}. Skipping.")
        except Exception as e:
            print(f"An unexpected error occurred while processing {filepath}: {e}. Skipping.")

    if not all_detailed_results:
        print("No data loaded. Exiting.")
        return

    print("\n--- Converting to Pandas DataFrame ---")
    df = pd.DataFrame(all_detailed_results)
    
    # Drop columns that are not scores or identifiers
    score_columns = [
        "jaccard_similarity",
        "block_level_tp",
        "block_level_fp",
        "block_level_fn",
        "block_level_precision",
        "block_level_recall",
        "block_level_f1_score"
    ]
    
    # Filter DataFrame to include only relevant columns for mean calculation
    numeric_score_df = df[score_columns].apply(pd.to_numeric, errors='coerce')
    
    # Re-add 'condition' for grouping
    numeric_score_df['condition'] = df['condition']

    print(f"DataFrame created with {len(df)} entries.")
    
    print("\n--- Computing Mean Scores per Condition ---")
    
    # Group by 'condition' and compute the mean for all numeric score columns
    mean_scores_per_condition = numeric_score_df.groupby('condition').mean()

    print("\n--- Mean Scores per Condition (DataFrame Visualization) ---")
    print(mean_scores_per_condition.round(4)) 

if __name__ == '__main__':
    analyze_and_visualize_scores(main_path)

/home/xmakaco/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-29 18:30:15.223670: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-29 18:30:15.223970: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-29 18:30:15.295243: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-29 18:30:15.511685: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is opt

--- Loading and Concatenating Data ---
Successfully loaded data for condition: img_only
Successfully loaded data for condition: json_only
Successfully loaded data for condition: img_json

--- Converting to Pandas DataFrame ---
DataFrame created with 60 entries.

--- Computing Mean Scores per Condition ---

--- Mean Scores per Condition (DataFrame Visualization) ---
           jaccard_similarity  block_level_tp  block_level_fp  block_level_fn  \
condition                                                                       
img_json               0.1632            4.30           12.15           17.85   
img_only               0.1495            3.55           15.05           18.60   
json_only              0.2158            4.20           11.95           17.95   

           block_level_precision  block_level_recall  block_level_f1_score  
condition                                                                   
img_json                  0.2703              0.2301                0.24

In [ ]:
# Raw analysis judge's evaluation


def construct_df(data_path):
    """Construct a pandas dataframe from the json file with judge evaluations"""
    with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)  

    rows = []
    for item in data:
        row = {
            "structure_id": item.get("structure_id"),
            "condition": item.get("condition"),
            "judge_evaluation": float(item.get("judge_evaluation", 0))
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    return df

data_path = os.path.join(main_path, "results", "all_dialogues_judged", "all_dialogues_judged.json")
df_judge = construct_df(data_path)

In [15]:
# Used to check how many structures in a condition received a certain score

df_judge[(df_judge["judge_evaluation"] == 4) & (df_judge["condition"] == "img_only")].value_counts()

structure_id              condition  judge_evaluation
C18_overlapping-reticles  img_only   4.0                 1
Name: count, dtype: int64

In [ ]:
# Data analysis for means of judge's evaluation

def compute_judge_score_means(json_path):
    
    with open(json_path, "r") as infile:
        data = json.load(infile)

    # Filter and collect rows with valid numeric judge evaluations
    rows = []
    for item in data:
        try:
            score = float(item["judge_evaluation"])
            rows.append({
                "structure_id": item["structure_id"],
                "condition": item["condition"],
                "judge_score": score
            })
        except (ValueError, TypeError):
            continue  # skip if not a valid number

    # Create a DataFrame
    df = pd.DataFrame(rows)

    # Compute means by condition
    means_by_condition = df.groupby("condition")["judge_score"].mean().reset_index()

    # Add overall mean as a row
    overall_mean = pd.DataFrame([{
        "condition": "Overall",
        "judge_score": df["judge_score"].mean()
    }])

    # Combine results
    result_df = pd.concat([means_by_condition, overall_mean], ignore_index=True)

    return result_df

# Example usage
result = compute_judge_score_means("/home/xmakaco/cimec/MLLMs-construction-company-main-new/results/all_dialogues_judged/all_dialogues_judged.json")
print(result)

   condition  judge_score
0   img_json     2.650000
1   img_only     2.800000
2  json_only     2.550000
3    Overall     2.666667
